In [1]:
import numpy as np
import pandas as pd

In [5]:
DATA_DIR = "/Users/jowanglin/regression-based_ERP/data/stimuli"
CODES_FILE_NAME = "JY_codes_list{}.txt"
WORDS_FILE_NAME = "JY_words_list{}.txt"

In [ ]:
def parse_elist(elist_inpath: str, drop_boundaries: bool=False) -> pd.DataFrame:
    with open(elist_inpath) as f:
        lines = f.readlines()
    for idx, line in enumerate(lines):
        if line.startswith("# item"):
            break
    lines = lines[idx:]
    elist_outpath = elist_inpath.split(".")[0] + "_reformatted.txt"
    with open(elist_outpath, "w") as f:
        f.writelines(lines)

    elist_df = pd.read_csv(elist_outpath, sep="\t")
    elist_df = elist_df.iloc[1:].reset_index(drop=True)
    elist_df = elist_df.rename(mapper=lambda s: s.replace(" ", ""), axis=1)
    elist_df = elist_df.rename(columns={"b_flags": "b_flags-a_flags",
                                        "a_flags": "enable",
                                        "enable": "bin",
                                        "bin": "drop"})
    elist_df = elist_df.drop(columns="drop")
    elist_df["ecode"] = elist_df["ecode"].astype(int)
    if drop_boundaries:
        elist_df = elist_df.loc[~ elist_df["label"].astype(str).str.contains("boundary",
                                                                 case=False,
                                                                 na=False,
                                                                 regex=False)].reset_index(drop=True)
    return elist_df


elist_df = parse_elist(f"{DATA_DIR}/jy_subj020_elist.txt")
display(elist_df)

ecodes = elist_df["ecode"].copy()
counts = dict(ecodes.value_counts())
condition_code_counts = np.array([counts[code] for code in range(241, 249)])
if np.all(np.diff(condition_code_counts) == 0):
    print(f"Condition code counts: {condition_code_counts}")

item_codes_indices = np.asarray(ecodes.loc[ecodes.isin(range(241, 249))].index) + 1
assert len(item_codes_indices) == sum(condition_code_counts)
item_codes = ecodes.iloc[item_codes_indices]
condition_codes = ecodes.iloc[item_codes_indices - 1].to_numpy()
if item_codes.is_unique:
    item_codes = item_codes.to_numpy()
    my_dict = dict()
    for code in set(condition_codes):
        codes_sorted = np.sort(item_codes[condition_codes == code])
        if np.all(np.diff(codes_sorted) == 1):
            val = f"{codes_sorted[0]}-{codes_sorted[-1]}"
            my_dict[int(code)] = val
        else:
            my_dict[int(code)] = codes_sorted

display(my_dict)



,#item,bepoch,ecode,label,onset,diff,dura,b_flags-a_flags,enable,bin
0,1,0.0,-99,boundary,0.0000,0.00,NaN,00000000 00000000,1.0,[ ]
1,2,0.0,254,S254,34.6250,34625.00,1.0,00000000 00000000,1.0,[ ]
2,3,0.0,1,S1,35.2090,584.00,1.0,00000000 00000000,1.0,[ ]
3,4,0.0,254,S254,45.8160,10607.00,1.0,00000000 00000000,1.0,[ ]
4,5,0.0,251,S251,46.3370,521.00,1.0,00000000 00000000,1.0,[ ]
...,...,...,...,...,...,...,...,...,...,...
3223,3224,0.0,253,S253,3239.5491,1802.00,1.0,00000000 00000000,1.0,[ ]
3224,3225,0.0,1,S1,3241.0640,1514.89,1.0,00000000 00000000,1.0,[ ]
3225,3226,0.0,254,S254,3242.7959,1731.93,1.0,00000000 00000000,1.0,[ ]
3226,3227,0.0,1,S1,3244.3149,1519.04,1.0,00000000 00000000,1.0,[ ]


Condition code counts: [28 28 28 28 28 28 28 28]


{241: '1-28',
 242: '29-56',
 243: '57-84',
 244: '85-112',
 245: '113-140',
 246: '141-168',
 247: '169-196',
 248: '197-224'}

In [ ]:
def parse_ecode_to_emo_dict(ecode_to_emo_dict: dict) -> dict:
    def helper(desc: str):
        desc_parsed = [s.split("-") for s in desc.split(";")]
        desc_parsed = [s for sublist in desc_parsed for s in range(int(sublist[0]), int(sublist[-1])+1)]
        assert len(desc_parsed) == len(set(desc_parsed))
        return desc_parsed
    return {key: helper(val) for key, val in ecode_to_emo_dict.items()}

ecode_to_emo_dict = dict(pos="3;4;14;18-22;25-28;29-30;33-37;39;43;45;50;53-54;57;62-69;77-78;85;90;92-97;105;107",
                    neg="1;2;5-13;15-17;23-24;31-32;35;38;40-42;44;46-49;51-52;55-56;58-61;70-76;79-84;86-89;91;98-99;100-104;106;108-112",
                    neu="113-224")
ecode_to_emo_dict = parse_ecode_to_emo_dict(ecode_to_emo_dict)

print(set(ecode_to_emo_dict["pos"]).issubset(set(list(range(1, 113)))))
print(set(ecode_to_emo_dict["neg"]).issubset(set(list(range(1, 113)))))
print(set(ecode_to_emo_dict["neu"]) - set(list(range(113, 225))) == set())
print(ecode_to_emo_dict)

True
True
True
{'pos': [3, 4, 14, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29, 30, 33, 34, 35, 36, 37, 39, 43, 45, 50, 53, 54, 57, 62, 63, 64, 65, 66, 67, 68, 69, 77, 78, 85, 90, 92, 93, 94, 95, 96, 97, 105, 107], 'neg': [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 23, 24, 31, 32, 35, 38, 40, 41, 42, 44, 46, 47, 48, 49, 51, 52, 55, 56, 58, 59, 60, 61, 70, 71, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 84, 86, 87, 88, 89, 91, 98, 99, 100, 101, 102, 103, 104, 106, 108, 109, 110, 111, 112], 'neu': [113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 21

In [84]:
def read_file(list_num: int, file_name: str) -> list[list]:
    file = open(f"{DATA_DIR}/{file_name.format(list_num)}", "r")
    contents = [line.replace("\n", "\t").split("\t") for line in file.readlines()]
    contents = [[w.strip() for w in line if w != ""] for line in contents]
    file.close()
    return contents

def label_emotion(codes: list[list], ecode_to_emo_dict: dict) -> list:
    def each_pair(sublist: list) -> str:
        if 245 <= int(sublist[1]) <= 248:
            if int(sublist[0]) in ecode_to_emo_dict["neu"]:
                return "neu"
            else:
                print(f"Inconsistency with event code scheme detected: {sublist}")
                return "-"
        else:
            if int(sublist[0]) in ecode_to_emo_dict["pos"]:
                return "pos"
            elif int(sublist[0]) in ecode_to_emo_dict["neg"]:
                return "neg"
            else:
                print(f"Inconsistency with event code scheme detected: {sublist}")
                return "-"  
    return [each_pair(sublist) for sublist in codes]


list_num = 1
words = read_file(list_num, WORDS_FILE_NAME)
codes = read_file(list_num, CODES_FILE_NAME)

emotions = label_emotion(codes, ecode_to_emo_dict)
my_df = pd.DataFrame({"emotion": emotions,
                      "length": [int(w[0]) for w in words],
                      "sentence": [w[1:] for w in words]})
display(my_df)

stats = np.stack([my_df.loc[my_df["emotion"]==emo]["length"].describe().to_numpy() for emo in ["pos", "neg", "neu"]])
stats_df = pd.DataFrame(data=stats.T, index=my_df["length"].describe().index, columns=["pos", "neg", "neu"]).T
stats_df["count"] = stats_df["count"].astype(int)
display(stats_df)

,emotion,length,sentence
0,pos,12,"[夫妻倆, 關係, 親密, 無間, 且, 深深, 相愛，, 是, 因為, 彼此, 願意, 經營。]"
1,neg,10,"[太宰治, 一生, 鬱鬱, 寡歡，, 其作品, 充滿著, 對, 世間, 的, 不滿。]"
2,neg,10,"[這次, 真的, 山窮, 水盡, 走投, 無路了，, 我, 感到, 徹底地, 絕望。]"
3,neu,7,"[下課後, 我會, 先去, 圖書館，, 晚上, 再去, 吃飯。]"
4,neu,10,"[遊客, 來到, 這邊，, 一定, 買, 當地, 手工, 織品, 作為, 擺設。]"
...,...,...,...
219,pos,11,"[畢業時, 回首, 這三年, 快樂的, 高中, 歲月，, 和, 好友們, 的回憶, 充滿了,..."
220,pos,11,"[伴侶, 送我, 手寫的, 暖心, 卡片，, 是我, 收過, 最有, 意義, 的, 手作。]"
221,neg,12,"[他, 不務, 正業, 又, 經常, 與, 黑道, 為伍，, 現在, 的工作, 是, 討債。]"
222,pos,10,"[如期, 抽到, 心中, 想要的, 贈品，, 大家, 直呼, 他, 真, 湊巧。]"


,count,mean,std,min,25%,50%,75%,max
pos,46,10.413043,1.146513,8.0,10.0,10.0,11.0,13.0
neg,66,10.378788,1.160440,7.0,10.0,10.0,11.0,13.0
neu,112,10.205357,1.382897,7.0,9.0,10.0,11.0,14.0


In [95]:
neu_df = my_df.loc[my_df["emotion"]=="neu"].copy()
neu_df = neu_df.reset_index(drop=True)
emo_df = my_df.loc[my_df["emotion"]!="neu"].copy()
emo_df = emo_df.reset_index(drop=True)
neu_df.to_excel(f"{DATA_DIR}/neu_frames.xlsx", index=False)
emo_df.to_excel(f"{DATA_DIR}/emo_frames.xlsx", index=False)

In [105]:
def detect_anomaly(frames: list[list]):
    def helper(sublist: list[str]):
        return [(i+1, x) for i, x in enumerate(sublist) if x.startswith("的")]
    to_return = [helper(sublist) for sublist in frames]
    return {i: (frames[i], lst) for i, lst in enumerate(to_return) if lst}

neu_frames = neu_df["sentence"].to_list()
neu_anomaly = detect_anomaly(neu_frames)
emo_frames = emo_df["sentence"].to_list()
emo_anomaly = detect_anomaly(emo_frames)


In [ ]:
neu_anomaly_filt = [tup for tup in sum([val[1]
                        for val in list(neu_anomaly.values())], [])
                        if tup[0] in {6, 7, 8}]

emo_anomaly_filt = [tup for tup in sum([val[1]
                        for val in list(emo_anomaly.values())], [])
                        if tup[0] in {6, 7, 8}]

print(len(neu_anomaly_filt))
print(len(emo_anomaly_filt))


10
10


In [115]:
len(neu_anomaly)

30